# Court Citation Concept Enrichment — Minimal LLM Runner

This is the canonical Kaggle runner for court citation enrichment.

Core design:

1. **Static parser / normalizer does all deterministic work**:
   - statute anchors
   - case anchors
   - legal-source/document/event/secondary-source separation
   - self/page-reference removal
   - conservative outcome correction
   - retrieval-view construction

2. **LLM only does non-static semantic descriptor extraction**:
   - legal area/domain/topic/subtopic/micro-topic
   - English legal concepts
   - exact original-language legal terms
   - doctrinal rule / legal test only if truly stated
   - fact-pattern tags
   - procedural context
   - paragraph role / authority role hints

The notebook intentionally does **not** ask the LLM for generated questions, summaries, final statute anchors, final case anchors, or final retrieval views.


In [ ]:
# Optional install cell for Kaggle
# Run once if vLLM/transformers are missing, then restart the kernel.

# !pip install -q -U "transformers>=4.45.0" accelerate safetensors pandas tqdm
# !pip install -q -U "vllm>=0.6.0" || true
# !pip uninstall -y flashinfer flashinfer-python || true


In [ ]:
# Cell 1 - Imports and configuration

from __future__ import annotations

import os
import re
import gc
import ast
import json
import time
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter

import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None


@dataclass
class Config:
    # Input data
    input_csv: str = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv"
    fallback_input_csv: str = "court_considerations.csv"

    # Normalizer code. Put court_enrichment_normalizer.py either beside this notebook,
    # in ./scripts, or in a Kaggle code dataset and update normalizer_search_paths.
    normalizer_search_paths: tuple[str, ...] = (
        ".",
        "./scripts",
        "/kaggle/working",
        "/kaggle/working/scripts",
        "/kaggle/input/code",
        "/kaggle/input/scripts",
    )

    # Local Kaggle model path
    model_name: str = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1"

    # Run controls
    output_dir: str = "/kaggle/working"
    start: int = 0
    limit: int = 50
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 120
    max_text_chars: int = 3200

    # GPU mode
    # single = one T4. Recommended for quality gates.
    # tp2 = one tensor-parallel model across both T4s. Use only if you want to test both GPUs in one notebook.
    gpu_mode: str = "single"

    # vLLM settings
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.78
    max_model_len: int = 4096
    max_num_seqs: int = 8
    batch_size: int = 4
    enforce_eager: bool = True
    quantization: str = "awq_marlin"
    disable_custom_all_reduce: bool = True
    force_triton_attention: bool = True

    # IMPORTANT: keep false for Kaggle vLLM v0.20 because structured_outputs caused
    # AttributeError("'dict' object has no attribute '_backend'").
    use_structured_outputs: bool = False

    # Generation: descriptor-only JSON should be short.
    max_new_tokens: int = 384
    retry_max_new_tokens: int = 512
    max_retries: int = 1
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False

cfg = Config()

# Apply GPU mode before vLLM import/init.
if cfg.gpu_mode == "single":
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == "tp2":
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True
else:
    raise ValueError("cfg.gpu_mode must be 'single' or 'tp2'")

if cfg.force_triton_attention:
    os.environ.setdefault("VLLM_ATTENTION_BACKEND", "TRITON_ATTN")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:256")

# Shard-safe output names.
out_end = cfg.start + cfg.limit - 1 if cfg.limit else cfg.start
cfg.output_jsonl = f"court_enriched_{cfg.start:07d}_{out_end:07d}.jsonl"
cfg.output_preview_csv = f"court_enriched_{cfg.start:07d}_{out_end:07d}_preview.csv"
cfg.output_failures_jsonl = f"court_enriched_{cfg.start:07d}_{out_end:07d}_failures.jsonl"
cfg.output_metrics_json = f"court_enriched_{cfg.start:07d}_{out_end:07d}_metrics.json"

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(asdict(cfg))

if torch is not None:
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            print(f"GPU {i}: {props.name}; free={free/1024**3:.2f} GiB / total={total/1024**3:.2f} GiB")


In [ ]:
# Cell 2 - Import deterministic normalizer

import sys

normalizer_found = None
for base in cfg.normalizer_search_paths:
    p = Path(base).resolve() / "court_enrichment_normalizer.py"
    if p.exists():
        normalizer_found = p
        sys.path.insert(0, str(p.parent))
        break

if normalizer_found is None:
    raise FileNotFoundError(
        "court_enrichment_normalizer.py not found. Put it beside this notebook, "
        "inside ./scripts, or attach it as a Kaggle dataset and add that path to cfg.normalizer_search_paths."
    )

from court_enrichment_normalizer import normalize_enriched_court_row

print("Loaded normalizer:", normalizer_found)


In [ ]:
# Cell 3 - Load CSV and select rows

def resolve_input_path(cfg: Config) -> Path:
    candidates = [
        Path(cfg.input_csv),
        Path(cfg.fallback_input_csv),
        Path('/kaggle/working') / cfg.fallback_input_csv,
        Path('/mnt/data') / cfg.fallback_input_csv,
        Path('/mnt/data/court_consideration.csv'),
        Path('/mnt/data/court_considerations.csv'),
    ]
    for p in candidates:
        if p.exists():
            return p
    if Path('/kaggle/input').exists():
        for pat in ['**/court_considerations.csv', '**/court_consideration.csv']:
            found = sorted(Path('/kaggle/input').glob(pat))
            if found:
                return found[0]
    raise FileNotFoundError('court_considerations.csv not found. Update cfg.input_csv.')


def pick_column(columns: List[str], preferred: List[str], contains_any: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in columns}
    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    for c in columns:
        lc = c.lower()
        if any(token in lc for token in contains_any):
            return c
    return None

input_path = resolve_input_path(cfg)
print('Using input:', input_path)

df = pd.read_csv(input_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))

citation_col = pick_column(list(df.columns), ['citation', 'cite', 'court_citation'], ['citation', 'cite', 'bge'])
text_col = pick_column(list(df.columns), ['text', 'consideration_text', 'paragraph_text', 'content', 'raw_text'], ['text', 'content', 'paragraph', 'consideration'])
if citation_col is None or text_col is None:
    raise ValueError(f'Could not infer citation/text columns. columns={list(df.columns)}')
print('citation_col:', citation_col)
print('text_col:', text_col)

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid['_text_len'] = valid[text_col].str.strip().str.len()
valid = valid[valid['_text_len'] >= cfg.min_text_chars].copy()

if cfg.sample_random:
    pool = valid.iloc[cfg.start:] if cfg.start else valid
    work_df = pool.sample(n=min(cfg.limit, len(pool)), random_state=cfg.random_seed)
else:
    end = cfg.start + cfg.limit if cfg.limit else None
    work_df = valid.iloc[cfg.start:end]

work_df = work_df.reset_index(drop=False).rename(columns={'index': '_source_row'})
print('Selected rows:', len(work_df))
display(work_df[[citation_col, text_col, '_text_len']].head(min(20, len(work_df))))


In [ ]:
# Cell 4 - Minimal LLM descriptor schema and prompt

# The LLM must NOT produce anchors, summaries, questions, or retrieval views.
# The normalizer owns anchors/outcomes/retrieval views.

LLM_DESCRIPTOR_SCHEMA = {
    "legal_area": "broad legal area, e.g. constitutional law, criminal procedure, tax law",
    "primary_domain": "stable high-level domain, e.g. political rights, pretrial detention",
    "secondary_domain": "narrower domain, e.g. official campaign intervention, collusion risk",
    "legal_domain_path": ["2-6 short labels from broad to narrow"],
    "topic": "short specific topic",
    "subtopic": "short narrower topic",
    "micro_topic": "most specific descriptor of the paragraph's legal issue",
    "concepts_en": ["3-8 precise English legal concepts"],
    "terms_original": ["3-10 exact important German/French/Italian terms from the text"],
    "doctrinal_rule": "only if this paragraph states a legal rule; otherwise empty string",
    "legal_test": "only if this paragraph states/applies a legal test; otherwise empty string",
    "fact_pattern_tags": ["0-6 concrete fact/procedure tags"],
    "procedural_context": "short procedural posture, if clear",
    "paragraph_role": "holding|reasoning|facts|procedural_history|legal_standard|application|citation|costs|notification|disposition|neutral",
    "authority_role": ["0-4 values such as legal_test, constitutional_standard, application_of_rule, factual_background, none"],
    "specificity_score": "number from 0 to 1"
}

FORBIDDEN_LLM_FIELDS = {
    'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
    'query_phrases_en', 'natural_language_queries', 'legal_question',
    'summary_en', 'english_summary', 'outcome_signal'
}

SYSTEM_PROMPT = """
You are a Swiss legal citation descriptor engine.

Return exactly one compact JSON object.
Do not use markdown.
Do not generate user questions.
Do not generate summaries.
Do not produce final statute_anchors or case_anchors.
Do not produce retrieval_views or normalized_anchors.
Do not infer final outcome_signal; deterministic code handles outcomes later.

Your task is only to extract semantic descriptors that static parsing cannot reliably infer:
legal area/domain, topic/subtopic/micro-topic, English legal concepts, exact original-language legal terms, doctrinal rule/test if stated, fact-pattern tags, procedural context, paragraph role, authority role, and specificity_score.

If the text is factual/procedural/boilerplate, keep doctrinal_rule and legal_test empty.
Use only information grounded in the text.
Keep arrays short and high-signal.
""".strip()

USER_TEMPLATE = """
Citation:
{citation}

Text:
{text}

Return JSON matching this descriptor schema only:
{schema}
""".strip()

REPAIR_TEMPLATE = """
The previous model output was invalid or had forbidden fields.

Citation:
{citation}

Text:
{text}

Previous output:
{bad_output}

Error:
{error}

Return exactly one complete compact JSON object matching this descriptor schema only:
{schema}

Forbidden fields: statute_anchors, case_anchors, normalized_anchors, retrieval_views, query_phrases_en, natural_language_queries, legal_question, summary_en, english_summary, outcome_signal.
""".strip()


def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text or '')).strip()
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()


def build_user_prompt(citation: str, text: str) -> str:
    return USER_TEMPLATE.format(
        citation=citation,
        text=trim_text(text, cfg.max_text_chars),
        schema=json.dumps(LLM_DESCRIPTOR_SCHEMA, ensure_ascii=False, indent=2),
    )


def build_repair_prompt(citation: str, text: str, bad_output: str, error: str) -> str:
    return REPAIR_TEMPLATE.format(
        citation=citation,
        text=trim_text(text, cfg.max_text_chars),
        bad_output=str(bad_output or '')[:2000],
        error=str(error)[:500],
        schema=json.dumps(LLM_DESCRIPTOR_SCHEMA, ensure_ascii=False, indent=2),
    )


In [ ]:
# Cell 5 - JSON parsing and minimal descriptor cleanup

def find_balanced_json_object(s: str) -> str:
    s = str(s or '').strip()
    s = re.sub(r'^\s*```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```\s*$', '', s)
    start = s.find('{')
    if start < 0:
        raise ValueError(f'No JSON object start found: {s[:400]}')
    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    return s[start:i+1]
    raise ValueError(f'No balanced JSON object found: {s[:800]}')


def parse_json_lenient(raw: str) -> Dict[str, Any]:
    js = find_balanced_json_object(raw)
    try:
        return json.loads(js)
    except Exception:
        pass
    cleaned = re.sub(r',\s*([}\]])', r'\1', js)
    cleaned = cleaned.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")
    try:
        return json.loads(cleaned)
    except Exception:
        pass
    return ast.literal_eval(cleaned)


def as_clean_list(x: Any, max_items: int) -> List[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, (list, tuple, set)):
        return []
    out, seen = [], set()
    for item in x:
        s = re.sub(r'\s+', ' ', str(item or '')).strip()
        if not s:
            continue
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def clean_descriptor(obj: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(obj, dict):
        obj = {}
    # Hard-strip forbidden fields even if the model emits them.
    for key in list(obj.keys()):
        if key in FORBIDDEN_LLM_FIELDS:
            obj.pop(key, None)

    out = {
        'legal_area': str(obj.get('legal_area') or '').strip()[:140],
        'primary_domain': str(obj.get('primary_domain') or '').strip()[:120],
        'secondary_domain': str(obj.get('secondary_domain') or '').strip()[:160],
        'topic': str(obj.get('topic') or '').strip()[:160],
        'subtopic': str(obj.get('subtopic') or '').strip()[:180],
        'micro_topic': str(obj.get('micro_topic') or '').strip()[:220],
        'legal_domain_path': as_clean_list(obj.get('legal_domain_path'), 6),
        'concepts_en': as_clean_list(obj.get('concepts_en'), 8),
        'terms_original': as_clean_list(obj.get('terms_original'), 10),
        'doctrinal_rule': str(obj.get('doctrinal_rule') or '').strip()[:420],
        'legal_test': str(obj.get('legal_test') or '').strip()[:320],
        'fact_pattern_tags': as_clean_list(obj.get('fact_pattern_tags'), 6),
        'procedural_context': str(obj.get('procedural_context') or '').strip()[:240],
        'paragraph_role': str(obj.get('paragraph_role') or 'neutral').strip(),
        'authority_role': as_clean_list(obj.get('authority_role') or ['none'], 4),
    }
    try:
        out['specificity_score'] = max(0.0, min(1.0, float(obj.get('specificity_score', 0.0))))
    except Exception:
        out['specificity_score'] = 0.0
    return out


def empty_descriptor(error: str = '') -> Dict[str, Any]:
    return {
        'legal_area': '',
        'primary_domain': '',
        'secondary_domain': '',
        'legal_domain_path': [],
        'topic': '',
        'subtopic': '',
        'micro_topic': '',
        'concepts_en': [],
        'terms_original': [],
        'doctrinal_rule': '',
        'legal_test': '',
        'fact_pattern_tags': [],
        'procedural_context': '',
        'paragraph_role': 'neutral',
        'authority_role': ['none'],
        'specificity_score': 0.0,
        '_descriptor_error': str(error)[:500],
    }


In [ ]:
# Cell 6 - vLLM engine, prompt-only JSON mode

class VllmDescriptorEngine:
    def __init__(self, cfg: Config):
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams

        self.cfg = cfg
        self.SamplingParams = SamplingParams
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)

        llm_kwargs = dict(
            model=cfg.model_name,
            trust_remote_code=True,
            tensor_parallel_size=cfg.tensor_parallel_size,
            gpu_memory_utilization=cfg.gpu_memory_utilization,
            max_model_len=cfg.max_model_len,
            max_num_seqs=cfg.max_num_seqs,
            enforce_eager=cfg.enforce_eager,
            quantization=cfg.quantization,
            disable_custom_all_reduce=cfg.disable_custom_all_reduce,
            disable_log_stats=True,
        )

        if cfg.force_triton_attention:
            try:
                from vllm.config import AttentionConfig
                try:
                    llm_kwargs['attention_config'] = AttentionConfig(backend='TRITON_ATTN')
                except Exception:
                    llm_kwargs['attention_config'] = AttentionConfig(backend='triton_attn')
                print('[vLLM] using explicit AttentionConfig backend=TRITON_ATTN')
            except Exception:
                print('[vLLM] AttentionConfig unavailable; relying on env/default backend')

        self.llm = LLM(**llm_kwargs)
        print('Engine: vLLM prompt-only JSON')
        print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '<all>'))
        print('tensor_parallel_size:', cfg.tensor_parallel_size)

    def sampling_params(self, max_tokens: int):
        return self.SamplingParams(
            temperature=self.cfg.temperature,
            top_p=self.cfg.top_p,
            max_tokens=max_tokens,
            repetition_penalty=self.cfg.repetition_penalty,
        )

    def chat_prompt(self, user_prompt: str) -> str:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ]
        try:
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=self.cfg.enable_thinking,
            )
        except TypeError:
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate(self, prompts: List[str], max_tokens: int) -> List[str]:
        full_prompts = [self.chat_prompt(p) for p in prompts]
        outputs = self.llm.generate(full_prompts, sampling_params=self.sampling_params(max_tokens), use_tqdm=False)
        texts = []
        for out in outputs:
            try:
                texts.append(out.outputs[0].text)
            except Exception:
                texts.append('')
        return texts

model_path = Path(cfg.model_name)
if not model_path.exists():
    raise FileNotFoundError(f'Model path does not exist: {model_path}. Attach the Kaggle model dataset.')

engine = VllmDescriptorEngine(cfg)


In [ ]:
# Cell 7 - Enrich rows: LLM descriptor -> deterministic normalizer -> final JSONL

def get_optional(row: pd.Series, key: str, default=None):
    return row[key] if key in row and pd.notna(row[key]) else default


def build_deterministic_metadata(row: pd.Series) -> Dict[str, Any]:
    # court_considerations.csv only has citation/text, but this supports richer v4 cards too.
    return {
        'citation': str(row[citation_col]),
        'text_excerpt_original': str(row[text_col]),
        'court_base': get_optional(row, 'court_base'),
        'legal_area': get_optional(row, 'legal_area'),
        'law_codes': get_optional(row, 'law_codes'),
        'statutes_cited': get_optional(row, 'statutes_cited'),
        'court_cases_cited': get_optional(row, 'court_cases_cited'),
        'authority_role': get_optional(row, 'authority_role'),
        'issue_labels_en': get_optional(row, 'issue_labels_en'),
        'matched_terms_multilingual': get_optional(row, 'matched_terms_multilingual'),
        'is_notification_paragraph': bool(get_optional(row, 'is_notification_paragraph', False)),
        'structural': get_optional(row, 'structural') or {},
    }


def generate_descriptor_for_row(row_dict: Dict[str, Any], first_raw: Optional[str] = None) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    citation = row_dict['citation']
    text = row_dict['text']
    raw = first_raw
    attempts = []

    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                raw = engine.generate([build_user_prompt(citation, text)], max_tokens=cfg.max_new_tokens)[0]
            obj = parse_json_lenient(raw)
            desc = clean_descriptor(obj)
            return desc, {'status': 'ok' if attempt == 0 else 'ok_after_retry', 'attempts': attempts, 'raw_output': raw}
        except Exception as exc:
            err = repr(exc)
            attempts.append({'attempt': attempt, 'error': err, 'raw_output': (raw or '')[:2000]})
            if attempt < cfg.max_retries:
                repair = build_repair_prompt(citation, text, raw or '', err)
                raw = engine.generate([repair], max_tokens=cfg.retry_max_new_tokens)[0]
            else:
                return empty_descriptor(err), {'status': 'descriptor_fallback', 'attempts': attempts, 'raw_output': raw or ''}


def build_final_record(row: pd.Series, descriptor: Dict[str, Any], gen_info: Dict[str, Any]) -> Dict[str, Any]:
    citation = str(row[citation_col])
    text = str(row[text_col])
    metadata = build_deterministic_metadata(row)

    normalized = normalize_enriched_court_row(
        citation=citation,
        text=text,
        llm_enrichment=descriptor,
        deterministic_metadata=metadata,
    )

    # Make generation status explicit while preserving normalizer quality fields.
    normalized['enrichment_quality']['llm_descriptor_status'] = gen_info['status']
    normalized['enrichment_quality']['question_generation_used'] = False
    normalized['enrichment_quality']['summary_generation_used'] = False
    normalized['enrichment_quality']['llm_generated_anchors'] = False

    record = {
        '_source_row': int(row['_source_row']),
        'citation': citation,
        'court_base': normalized['normalized_anchors']['self_references'][1]
            if len(normalized['normalized_anchors'].get('self_references', [])) > 1
            else citation,
        'source_family': 'court',
        'text': text,
        'rag_enrichment': normalized['rag_enrichment'],
        'normalized_anchors': normalized['normalized_anchors'],
        'anchor_quality_flags': normalized['anchor_quality_flags'],
        'retrieval_views': normalized['retrieval_views'],
        'enrichment_quality': normalized['enrichment_quality'],
    }

    if gen_info['status'] == 'descriptor_fallback':
        record['_debug_attempts'] = gen_info.get('attempts', [])
    return record

out_dir = Path(cfg.output_dir)
out_jsonl = out_dir / cfg.output_jsonl
out_csv = out_dir / cfg.output_preview_csv
out_failures = out_dir / cfg.output_failures_jsonl
out_metrics = out_dir / cfg.output_metrics_json

records = []
failure_records = []
t0 = time.time()

for start_i in tqdm(range(0, len(work_df), cfg.batch_size), desc='enrich batches'):
    batch = work_df.iloc[start_i:start_i + cfg.batch_size]
    row_dicts = [
        {'citation': str(row[citation_col]), 'text': str(row[text_col])}
        for _, row in batch.iterrows()
    ]
    prompts = [build_user_prompt(r['citation'], r['text']) for r in row_dicts]

    try:
        raws = engine.generate(prompts, max_tokens=cfg.max_new_tokens)
    except Exception as batch_exc:
        print('Batch generation failed; falling back to row-by-row:', repr(batch_exc))
        raws = [None] * len(row_dicts)

    for (_, row), row_dict, raw in zip(batch.iterrows(), row_dicts, raws):
        descriptor, gen_info = generate_descriptor_for_row(row_dict, first_raw=raw)
        rec = build_final_record(row, descriptor, gen_info)
        records.append(rec)
        if gen_info['status'] == 'descriptor_fallback':
            failure_records.append(rec)

elapsed = time.time() - t0

with out_jsonl.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

preview_rows = []
for rec in records:
    enr = rec['rag_enrichment']
    anchors = rec['normalized_anchors']
    quality = rec['enrichment_quality']
    views = rec['retrieval_views']
    preview_rows.append({
        '_source_row': rec['_source_row'],
        'citation': rec['citation'],
        'court_base': rec['court_base'],
        'descriptor_status': quality.get('llm_descriptor_status'),
        'legal_area': enr.get('legal_area'),
        'primary_domain': enr.get('primary_domain'),
        'secondary_domain': enr.get('secondary_domain'),
        'topic': enr.get('topic'),
        'subtopic': enr.get('subtopic'),
        'micro_topic': enr.get('micro_topic'),
        'concepts_en': ' | '.join(enr.get('concepts_en') or []),
        'terms_original': ' | '.join(enr.get('terms_original') or []),
        'statute_anchors': ' | '.join(anchors.get('statute_anchors') or []),
        'case_anchors': ' | '.join(anchors.get('case_anchors') or []),
        'legal_source_anchors': ' | '.join(anchors.get('legal_source_anchors') or []),
        'secondary_sources': ' | '.join(anchors.get('secondary_sources') or []),
        'document_or_plan_anchors': ' | '.join(anchors.get('document_or_plan_anchors') or []),
        'event_anchors': ' | '.join(anchors.get('event_anchors') or []),
        'paragraph_role': enr.get('paragraph_role'),
        'authority_role': ' | '.join(enr.get('authority_role') or []),
        'outcome_signal': enr.get('outcome_signal'),
        'specificity_score': enr.get('specificity_score'),
        'anchor_cleanup_count': quality.get('anchor_cleanup_count'),
        'semantic_concepts_en': views.get('semantic_concepts_en'),
    })

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(out_csv, index=False)

if failure_records:
    with out_failures.open('w', encoding='utf-8') as f:
        for rec in failure_records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
elif out_failures.exists():
    out_failures.unlink()

# Metrics
forbidden = {'statute_anchors', 'case_anchors', 'query_phrases_en', 'natural_language_queries', 'legal_question', 'summary_en', 'english_summary', 'retrieval_views', 'normalized_anchors', 'outcome_signal'}

def find_forbidden_in_rag(rec):
    return sorted(set((rec.get('rag_enrichment') or {}).keys()) & forbidden)

metrics = {
    'start': cfg.start,
    'limit': cfg.limit,
    'processed': len(records),
    'elapsed_seconds': round(elapsed, 3),
    'rows_per_second': len(records) / max(elapsed, 1e-9),
    'descriptor_status_counts': dict(Counter(r['enrichment_quality'].get('llm_descriptor_status') for r in records)),
    'forbidden_rag_field_rows': sum(bool(find_forbidden_in_rag(r)) for r in records),
    'self_anchor_rows': sum(r['citation'] in (r['normalized_anchors'].get('case_anchors') or []) for r in records),
    'fallback_rows': len(failure_records),
    'output_jsonl': str(out_jsonl),
    'output_preview_csv': str(out_csv),
}
out_metrics.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(metrics, indent=2))
print('JSONL:', out_jsonl)
print('Preview CSV:', out_csv)
print('Metrics:', out_metrics)
if failure_records:
    print('Descriptor fallback rows:', out_failures)

display(preview_df.head(20))


In [ ]:
# Cell 8 - QC checks

FORBIDDEN_FINAL_RAG_FIELDS = {
    'statute_anchors', 'case_anchors', 'query_phrases_en', 'natural_language_queries',
    'legal_question', 'summary_en', 'english_summary', 'retrieval_views', 'normalized_anchors'
}

qc_rows = []
for rec in records:
    enr = rec.get('rag_enrichment') or {}
    anchors = rec.get('normalized_anchors') or {}
    qc_rows.append({
        'citation': rec.get('citation'),
        'descriptor_status': rec.get('enrichment_quality', {}).get('llm_descriptor_status'),
        'forbidden_rag_fields': sorted(set(enr.keys()) & FORBIDDEN_FINAL_RAG_FIELDS),
        'concept_count': len(enr.get('concepts_en') or []),
        'terms_count': len(enr.get('terms_original') or []),
        'statute_count': len(anchors.get('statute_anchors') or []),
        'case_count': len(anchors.get('case_anchors') or []),
        'self_in_case_anchors': rec.get('citation') in (anchors.get('case_anchors') or []),
        'rule_suppressed': rec.get('enrichment_quality', {}).get('rule_fields_suppressed'),
        'anchor_cleanup_count': rec.get('enrichment_quality', {}).get('anchor_cleanup_count'),
        'specificity_score': enr.get('specificity_score'),
    })

qc_df = pd.DataFrame(qc_rows)
display(qc_df)

print('Rows:', len(qc_df))
print('Forbidden rag field rows:', int(qc_df['forbidden_rag_fields'].apply(bool).sum()))
print('Self in case anchors:', int(qc_df['self_in_case_anchors'].sum()))
print('Descriptor fallback rows:', int((qc_df['descriptor_status'] == 'descriptor_fallback').sum()))
print('Average concepts:', float(qc_df['concept_count'].mean()) if len(qc_df) else 0)
print('Average terms:', float(qc_df['terms_count'].mean()) if len(qc_df) else 0)


In [ ]:
# Cell 9 - Inspect one final JSON record

if records:
    print(json.dumps(records[0], ensure_ascii=False, indent=2)[:9000])
else:
    print('No records produced.')
